<a href="https://colab.research.google.com/github/Bilal140202/TalkDrive_by_BilalAnsari/blob/main/TalkDrive_by_BilalAnsari.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(120deg, #0f2027, #203a43, #2c5364); border-radius: 18px; padding: 36px 28px; margin-bottom: 8px; box-shadow: 0 12px 40px rgba(0,0,0,0.5);">
  <div style="display:flex; align-items:center; gap:16px; flex-wrap:wrap; justify-content:center;">
    <span style="font-size:3.2em;">🗣️</span>
    <div>
      <h1 style="margin:0; color:#ffffff; font-size:2.1em; letter-spacing:-0.5px;">TalkDrive — AI Talking Head Runner</h1>
      <p style="margin:6px 0 0; color:#94b8c8; font-size:1.05em;">Audio-to-Portrait Animation · Powered by SoulX FlashHead 1.3B · Google Colab T4</p>
    </div>
  </div>
  <hr style="border:none; border-top:1px solid rgba(255,255,255,0.12); margin:20px 0 16px;">
  <div style="display:flex; flex-wrap:wrap; justify-content:center; gap:10px;">
    <a href="https://ansaribilal.com" target="_blank" style="background:#00c9a7; color:#000; padding:7px 18px; border-radius:8px; text-decoration:none; font-weight:700; font-size:0.9em;">🌐 ansaribilal.com</a>
    <img src="https://img.shields.io/badge/Colab-T4%20GPU-F9AB00?style=flat-square&logo=googlecolab&logoColor=white" style="height:32px;" />
    <img src="https://img.shields.io/badge/Made%20by-Bilal%20Ansari-2c5364?style=flat-square" style="height:32px;" />
  </div>
</div>

---

## 📖 What Is This?

**TalkDrive** is a Colab runner for **SoulX FlashHead** — an open-source 1.3B diffusion model that animates a still portrait photo using an audio clip.  
This notebook is **stabilised, documented, and extended** by **Bilal Ansari** ([ansaribilal.com](https://ansaribilal.com)) to fix Colab compatibility issues and add 16:9 output compositing.

---

## 🧠 Architecture Overview

```
Portrait photo  ──►┐
                   │  SoulX FlashHead 1.3B (DiT + Wav2Vec2)  ──►  512×512 animated clip
Audio / speech  ──►┘
                                                                        │
                                                             16:9 Compositor
                                                                        │
                                                             Final video (original resolution)
```

| Component | Role |
|---|---|
| **SoulX FlashHead 1.3B** | Diffusion transformer: generates talking-head frames conditioned on audio embeddings |
| **Wav2Vec2-base-960h** | Facebook speech model — extracts phoneme features that drive lip-sync |
| **CPUFaceHandler (OpenCV)** | Detects & crops face before inference (replaces broken mediapipe) |
| **16:9 Compositor** | Pastes the 512×512 generated clip back into the full-res original |
| **Gradio UI** | Browser interface with public shareable link |

---

## ⚙️ Lite vs Pro

| | Lite | Pro |
|---|---|---|
| **Speed** | Fast | Slow |
| **Quality** | Good | High |
| **Recommended for** | Testing | Final render |

---

## 🚦 Before You Run

> ‼️ **Go to Runtime → Change runtime type → T4 GPU first.**  
> Then run cells **top to bottom, one at a time.**


In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  STAGE 1 — Environment & Hardware Verification                  │
# │                                                                 │
# │  Confirms the Colab runtime has a GPU and that PyTorch can      │
# │  actually allocate tensors on it. A dummy CUDA tensor is        │
# │  allocated and immediately freed as a smoke test.               │
# │                                                                 │
# │  Expected: ✅ PyTorch CUDA is working!                          │
# │  If ❌: stop here, switch to T4 GPU runtime, rerun.             │
# └─────────────────────────────────────────────────────────────────┘

# @title 🔎 Stage 1 · Environment & Hardware Check

print("━" * 55)
print("  TalkDrive — Environment Verification")
print("  Bilal Ansari · ansaribilal.com")
print("━" * 55)

print("\n📡 GPU report:")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
print()

import torch

print(f"  PyTorch version : {torch.__version__}")
print(f"  CUDA available  : {torch.cuda.is_available()}")
print(f"  CUDA toolkit    : {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"  GPU             : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    try:
        probe = torch.zeros(1, device="cuda")
        del probe
        print("\n  ✅ PyTorch CUDA is working!")
    except Exception as gpu_err:
        print(f"\n  ⚠️  CUDA tensor test failed: {gpu_err}")
else:
    print("\n  ❌ No GPU found — Runtime → Change runtime type → T4 GPU")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TalkDrive — Environment Verification
  Bilal Ansari · ansaribilal.com
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📡 GPU report:
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07

  PyTorch version : 2.10.0+cu128
  CUDA available  : True
  CUDA toolkit    : 12.8
  GPU             : Tesla T4
  VRAM            : 14.6 GB

  ✅ PyTorch CUDA is working!


In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  STAGE 2 — Repository Clone & System Dependencies               │
# │                                                                 │
# │  • Installs ffmpeg via apt (system binary used by video encode) │
# │  • Clones Soul-AILab/SoulX-FlashHead into /content/             │
# │  • git-pulls if the repo already exists from a previous run     │
# └─────────────────────────────────────────────────────────────────┘

# @title 📦 Stage 2 · Clone Repository & Install System Binaries

import os

print("🔧 Installing system dependency: ffmpeg")
!apt-get install -y ffmpeg -q

REPO_PATH = "/content/SoulX-FlashHead"

print("\n📥 Fetching SoulX-FlashHead source code...")
if os.path.exists(REPO_PATH):
    print("  → Already present, pulling latest changes...")
    %cd /content/SoulX-FlashHead
    !git pull
else:
    !git clone https://github.com/Soul-AILab/SoulX-FlashHead.git
    %cd /content/SoulX-FlashHead

print("\n📂 Repo contents:")
!ls -1
print("\n✅ System setup complete.")


🔧 Installing system dependency: ffmpeg
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.

📥 Fetching SoulX-FlashHead source code...
Cloning into 'SoulX-FlashHead'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 155 (delta 33), reused 26 (delta 18), pack-reused 90 (from 1)
Receiving objects: 100% (155/155), 6.53 MiB | 36.13 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/SoulX-FlashHead

📂 Repo contents:
assets
examples
flash_head
generate_video.py
gradio_app.py
gradio_app_streaming.py
inference_script_multi_gpu_pro.sh
inference_script_single_gpu_lite.sh
inference_script_single_gpu_pro.sh
LICENSE
README.md
requirements.txt

✅ System setup complete.


In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  STAGE 3 — Python Package Installation (~2–4 minutes)           │
# │                                                                 │
# │  Key strategy: do NOT reinstall torch.                          │
# │  Colab ships a CUDA-tuned torch build. Overwriting it breaks    │
# │  GPU access. All packages are installed around the existing     │
# │  torch, not over it.                                            │
# │                                                                 │
# │  Package groups:                                                │
# │   [1] xformers — memory-efficient attention (speeds up DiT)     │
# │   [2] Core ML  — diffusers, transformers, accelerate            │
# │   [3] Vision & Audio — opencv, librosa, decord, mediapipe       │
# │   [4] UI & utils — gradio, loguru, tqdm, imageio               │
# │   [5] xfuser — optional distributed inference helpers           │
# └─────────────────────────────────────────────────────────────────┘

# @title ⚙️ Stage 3 · Install Python Packages

import torch  # imported early so we confirm it survived

print("━" * 55)
print("  Installing packages (torch is NOT reinstalled)")
print("━" * 55)

print("\n[1/5] xformers — memory-efficient attention kernels")
!pip install -q xformers

print("\n[2/5] Core ML stack — diffusers / transformers / accelerate")
!pip install -q \
    "opencv-python-headless>=4.12.0.88" \
    "diffusers>=0.34.0" \
    "transformers==4.57.3" \
    "tokenizers>=0.20.3" \
    "accelerate>=1.8.1" \
    tqdm imageio easydict ftfy \
    "imageio-ffmpeg" \
    scikit-image \
    loguru \
    "gradio>=5.0.0" \
    pyloudnorm \
    decord \
    librosa \
    "mediapipe>=0.10.13" \
    flask \
    huggingface_hub

print("\n[3/5] xfuser + helpers — distributed inference support")
!pip install -q "xfuser>=0.4.3" || pip install -q "xfuser>=0.4.3" --no-deps
!pip install -q yunchang distvae sentencepiece beautifulsoup4 einops 2>/dev/null || true

print("\n━" * 28)
print("  Post-install verification")
print("━" * 55)
print(f"  torch      : {torch.__version__}  (CUDA={torch.cuda.is_available()})")
for pkg_name, import_name in [
    ("xformers",  "xformers"),
    ("xfuser",    "xfuser"),
    ("diffusers", "diffusers"),
    ("gradio",    "gradio"),
    ("librosa",   "librosa"),
]:
    try:
        mod = __import__(import_name)
        print(f"  {pkg_name:<12}: {mod.__version__}")
    except Exception as e:
        print(f"  {pkg_name:<12}: ⚠️  {e}")

print("\n✅ Package installation complete.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Installing packages (torch is NOT reinstalled)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[1/5] xformers — memory-efficient attention kernels
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.8 MB/s eta 0:00:00

[2/5] Core ML stack — diffusers / transformers / accelerate
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.8 MB/s eta 0:00:00

[3/5] xfuse

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


  xfuser      : ⚠️  module 'xfuser' has no attribute '__version__'
  diffusers   : 0.37.1
  gradio      : 5.50.0
  librosa     : 0.11.0

✅ Package installation complete.


In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  STAGE 4 — Model Weight Download (~5.4 GB total)                │
# │                                                                 │
# │  Model 1: SoulX-FlashHead-1_3B  (Soul-AILab)                   │
# │    The diffusion transformer checkpoint. Includes denoising     │
# │    weights, motion embeddings, and all components needed to     │
# │    generate animated portrait frames. Size: ~5 GB              │
# │                                                                 │
# │  Model 2: wav2vec2-base-960h  (Facebook / Meta)                 │
# │    Speech model trained on 960h of LibriSpeech audio.           │
# │    Extracts frame-level phoneme features that drive lip-sync    │
# │    and facial expressions. Size: ~360 MB                        │
# │                                                                 │
# │  Both go to /content/models/ and survive tab reloads within     │
# │  the same Colab session.                                        │
# └─────────────────────────────────────────────────────────────────┘

# @title 🗂️ Stage 4 · Download Model Weights from HuggingFace

import os

MODELS_ROOT = "/content/models"
CKPT_DIR    = f"{MODELS_ROOT}/SoulX-FlashHead-1_3B"
WAV2VEC_DIR = f"{MODELS_ROOT}/wav2vec2-base-960h"

os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(WAV2VEC_DIR, exist_ok=True)

print("📥 [1/2] SoulX-FlashHead-1_3B")
print(f"   Destination: {CKPT_DIR}")
!huggingface-cli download Soul-AILab/SoulX-FlashHead-1_3B \
    --local-dir {CKPT_DIR} 2>&1 | grep -E "Fetching|Downloading|100%|%\|"

print("\n📥 [2/2] wav2vec2-base-960h  (Facebook / Meta)")
print(f"   Destination: {WAV2VEC_DIR}")
!huggingface-cli download facebook/wav2vec2-base-960h \
    --local-dir {WAV2VEC_DIR} 2>&1 | grep -E "Fetching|Downloading|100%|%\|"

print("\n── Download verification ──")
for label, path in [("FlashHead 1.3B", CKPT_DIR), ("Wav2Vec2-960h", WAV2VEC_DIR)]:
    if os.path.exists(path) and len(os.listdir(path)) > 0:
        n = len(os.listdir(path))
        print(f"  ✅ {label}  ({n} files)")
    else:
        print(f"  ❌ {label}  — MISSING, check download output above")


📥 [1/2] SoulX-FlashHead-1_3B
   Destination: /content/models/SoulX-FlashHead-1_3B
Fetching 18 files: 100%|██████████| 18/18 [03:44<00:00, 12.49s/it]

📥 [2/2] wav2vec2-base-960h  (Facebook / Meta)
   Destination: /content/models/wav2vec2-base-960h
Fetching 11 files: 100%|██████████| 11/11 [00:12<00:00,  1.11s/it]

── Download verification ──
  ✅ FlashHead 1.3B  (10 files)
  ✅ Wav2Vec2-960h  (12 files)


In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  STAGE 5 — Source Patching (3 files modified)                   │
# │                                                                 │
# │  PATCH A — cpu_face_handler.py                                  │
# │    Replaces mediapipe face detection with OpenCV DNN/Haar.      │
# │    mediapipe's native libs conflict with Colab's CUDA runtime.  │
# │    OpenCV DNN uses ResNet-SSD for accurate detection; Haar      │
# │    cascade is the fallback. Output format (relative coords)     │
# │    is identical so the rest of the pipeline is unaffected.      │
# │                                                                 │
# │  PATCH B — facecrop.py                                          │
# │    Adds bbox persistence: after the face is cropped, the exact  │
# │    pixel coordinates are written to a temp JSON file that the   │
# │    compositor reads to know where to paste the output clip.     │
# │                                                                 │
# │  PATCH C — gradio_app.py                                        │
# │    • Absolute /content/ model paths (relative paths break)      │
# │    • Default mode → Lite                                        │
# │    • Face-crop default → ON                                     │
# │    • share=True for public Gradio URL                           │
# │    • Injects _composite_to_original() and wires it into both    │
# │      run_inference() and run_multi_gpu_inference()              │
# └─────────────────────────────────────────────────────────────────┘

# @title 🔧 Stage 5 · Apply Compatibility & Compositing Patches
# @markdown Patches three source files. Safe to re-run (git reset applied first).

import os, re

os.chdir("/content/SoulX-FlashHead")

# Reset to clean upstream state before applying patches
print("🔄 Resetting files to upstream state...")
!git checkout -- gradio_app.py 2>/dev/null
!git checkout -- flash_head/utils/facecrop.py 2>/dev/null
!git checkout -- flash_head/utils/cpu_face_handler.py 2>/dev/null

# ═══════════════════════════════════════════════════
# PATCH A: Rewrite CPUFaceHandler — OpenCV only, no mediapipe
# ═══════════════════════════════════════════════════
HANDLER_PATH = "/content/SoulX-FlashHead/flash_head/utils/cpu_face_handler.py"

handler_src = '''import cv2
import numpy as np
from typing import Tuple, List

class CPUFaceHandler:
    # Drop-in replacement for the mediapipe-based handler.
    # Returns bounding boxes as relative coordinates [x1,y1,x2,y2] in [0,1],
    # preserving the exact interface expected by facecrop.py.
    #
    # Detection priority:
    #   1. OpenCV DNN with ResNet-SSD  (auto-downloaded, higher accuracy)
    #   2. Haar cascade fallback        (always available in cv2)

    def __init__(self, model_selection: int = 1, min_detection_confidence: float = 0.0):
        import urllib.request

        proto_path  = "/content/SoulX-FlashHead/deploy.prototxt"
        model_path  = "/content/SoulX-FlashHead/res10_300x300_ssd_iter_140000.caffemodel"
        self.use_dnn  = False
        self.use_haar = False

        if not os.path.exists(proto_path):
            try:
                urllib.request.urlretrieve(
                    "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
                    proto_path)
            except Exception:
                pass

        if not os.path.exists(model_path):
            try:
                urllib.request.urlretrieve(
                    "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel",
                    model_path)
            except Exception:
                pass

        if os.path.exists(proto_path) and os.path.exists(model_path):
            self.net = cv2.dnn.readNetFromCaffe(proto_path, model_path)
            self.use_dnn = True
        else:
            cascade_xml = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
            self.cascade = cv2.CascadeClassifier(cascade_xml)
            self.use_haar = True

    def detect(self, image: np.ndarray) -> Tuple[List, List]:
        # Returns (bboxes, scores) with bboxes in relative [0,1] coordinates
        bboxes, scores = [], []
        img_h, img_w = image.shape[:2]

        if self.use_dnn:
            blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300), (104.0, 177.0, 123.0))
            self.net.setInput(blob)
            detections = self.net.forward()
            for i in range(detections.shape[2]):
                conf = float(detections[0, 0, i, 2])
                if conf > 0.5:
                    x1 = detections[0, 0, i, 3]
                    y1 = detections[0, 0, i, 4]
                    x2 = detections[0, 0, i, 5]
                    y2 = detections[0, 0, i, 6]
                    bboxes.append([float(x1), float(y1), float(x2), float(y2)])
                    scores.append(conf)

        elif self.use_haar:
            gray  = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
            faces = self.cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))
            for (x, y, w, h) in faces:
                bboxes.append([x/img_w, y/img_h, (x+w)/img_w, (y+h)/img_h])
                scores.append(1.0)

        return bboxes, scores

    def __call__(self, image: np.ndarray) -> Tuple[List, List]:
        return self.detect(image)
'''

with open(HANDLER_PATH, "w") as fh:
    fh.write("import os\n" + handler_src)
print("✅ PATCH A — CPUFaceHandler rewritten (OpenCV DNN/Haar, no mediapipe)")

# ═══════════════════════════════════════════════════
# PATCH B: facecrop.py — write crop bbox to disk for compositor
# ═══════════════════════════════════════════════════
FACECROP_PATH = "/content/SoulX-FlashHead/flash_head/utils/facecrop.py"

with open(FACECROP_PATH, "r") as fh:
    fc_src = fh.read()

# Add imports and module-level sentinel variable
fc_src = fc_src.replace(
    'import os',
    'import os\nimport json\n\n_LAST_CROP_BBOX = None  # populated during inference, read by compositor'
)

# After the face crop, persist exact bbox + image dims to a temp JSON file
fc_src = fc_src.replace(
    '    crop_face = face_image.crop(scaled_bbox)',
    '    global _LAST_CROP_BBOX\n'
    '    _LAST_CROP_BBOX = scaled_bbox + [img_w, img_h]\n'
    '    try:\n'
    '        with open("/tmp/_talkdrive_crop_bbox.json", "w") as _bbox_file:\n'
    '            json.dump({"bbox": scaled_bbox, "img_w": img_w, "img_h": img_h}, _bbox_file)\n'
    '    except Exception:\n'
    '        pass  # non-fatal; compositor will skip if file absent\n'
    '    crop_face = face_image.crop(scaled_bbox)'
)

with open(FACECROP_PATH, "w") as fh:
    fh.write(fc_src)
print("✅ PATCH B — facecrop.py now persists bbox to /tmp/_talkdrive_crop_bbox.json")

# ═══════════════════════════════════════════════════
# PATCH C: gradio_app.py — paths, defaults, branding, compositor
# ═══════════════════════════════════════════════════
GRADIO_PATH = "/content/SoulX-FlashHead/gradio_app.py"

with open(GRADIO_PATH, "r") as fh:
    ga_src = fh.read()

# Fix relative → absolute model paths
ga_src = ga_src.replace('value="models/SoulX-FlashHead-1_3B"', 'value="/content/models/SoulX-FlashHead-1_3B"')
ga_src = ga_src.replace('value="models/wav2vec2-base-960h"',   'value="/content/models/wav2vec2-base-960h"')

# Default mode to lite
ga_src = ga_src.replace('value="pro",', 'value="lite",')

# Face crop defaults ON (compositor needs the bbox file it produces)
ga_src = ga_src.replace(
    'label="Use Face Crop",\n                value=False,',
    'label="Use Face Crop",\n                value=True,'
)

# Enable public Gradio share link
if 'app.launch(share=True' not in ga_src:
    ga_src = ga_src.replace('app.launch()', 'app.launch(share=True, debug=True)')

# Replace the title Markdown header with branded HTML
hdr_match = re.search(r'^( *)gr\.Markdown\("# ⚡ SoulX-FlashHead Video Generator"\)', ga_src, re.MULTILINE)
if hdr_match:
    indent = hdr_match.group(1)
    brand_html = (
        '<div style="text-align:center; padding:22px 16px; '
        'background:linear-gradient(120deg,#0f2027,#203a43,#2c5364); '
        'border-radius:14px; box-shadow:0 8px 24px rgba(0,0,0,0.4); margin-bottom:18px;">'
        '<h1 style="font-size:2em; margin:0 0 6px; color:#ffffff;">🗣️ TalkDrive — AI Talking Head</h1>'
        '<p style="color:#94b8c8; margin:0 0 14px; font-size:1em;">'
        'SoulX FlashHead 1.3B · Audio-Driven Portrait Animation</p>'
        '<a href="https://ansaribilal.com" target="_blank" style="background:#00c9a7; color:#000; '
        'padding:7px 18px; border-radius:8px; text-decoration:none; font-weight:700; font-size:0.88em;">'
        '🌐 ansaribilal.com</a>'
        '<p style="margin-top:12px; font-size:0.8em; color:rgba(255,255,255,0.5);">'
        'Runner by <strong>Bilal Ansari</strong></p>'
        '</div>'
    )
    ga_src = ga_src.replace(
        hdr_match.group(0),
        indent + 'gr.HTML(' + repr(brand_html) + ')'
    )
    print("✅ PATCH C1 — Gradio header replaced with TalkDrive branding")

# Wire compositor into run_inference()
inf_match = re.search(
    r'(logger\.info\(f"Saved to \{final_video_path\}"\))\s*\n(\s*)(return final_video_path)',
    ga_src
)
if inf_match:
    ind = inf_match.group(2)
    ga_src = ga_src.replace(
        inf_match.group(0),
        inf_match.group(1) + '\n'
        + ind + '# Composite the 512x512 output back into the full-res original\n'
        + ind + 'try:\n'
        + ind + '    final_video_path = _composite_to_original(final_video_path, cond_image)\n'
        + ind + 'except Exception as comp_err:\n'
        + ind + '    import traceback; logger.error(f"COMPOSITOR ERROR: {comp_err}\\n{traceback.format_exc()}")\n'
        + ind + 'return final_video_path'
    )
    print("✅ PATCH C2 — Compositor wired into run_inference()")

# Wire compositor into run_multi_gpu_inference()
mgpu_match = re.search(
    r'(if os\.path\.exists\(save_path\):)\s*\n(\s*)(return save_path)',
    ga_src
)
if mgpu_match:
    ind2 = mgpu_match.group(2)
    ga_src = ga_src.replace(
        mgpu_match.group(0),
        mgpu_match.group(1) + '\n'
        + ind2 + '# Composite the 512x512 output back into the full-res original\n'
        + ind2 + 'try:\n'
        + ind2 + '    save_path = _composite_to_original(save_path, cond_image)\n'
        + ind2 + 'except Exception as comp_err:\n'
        + ind2 + '    import traceback; logger.error(f"COMPOSITOR ERROR: {comp_err}\\n{traceback.format_exc()}")\n'
        + ind2 + 'return save_path'
    )
    print("✅ PATCH C3 — Compositor wired into run_multi_gpu_inference()")

# Insert compositor function before the first def in the file
first_def_idx = ga_src.index("\ndef ")

compositor_fn = '''
def _composite_to_original(video_path: str, original_image_path: str) -> str:
    # Reads the face-crop bbox saved by facecrop.py and blends each
    # generated 512x512 frame back into the corresponding region of the
    # full-resolution original image.
    #
    # Pipeline:
    #   1. Load crop bbox from /tmp/_talkdrive_crop_bbox.json
    #   2. Skip if original is ~square (compositing adds no value)
    #   3. Per frame: resize → feather-blend → write to canvas
    #   4. Re-mux audio from generated clip into composite video
    import cv2, json, numpy as np, subprocess

    BBOX_FILE = "/tmp/_talkdrive_crop_bbox.json"

    if not os.path.exists(BBOX_FILE):
        logger.info("Crop bbox file absent — face_crop likely disabled, skipping compositor")
        return video_path

    with open(BBOX_FILE, "r") as bf:
        crop_data = json.load(bf)

    crop_x1, crop_y1, crop_x2, crop_y2 = crop_data["bbox"]
    orig_w = crop_data["img_w"]
    orig_h = crop_data["img_h"]
    os.remove(BBOX_FILE)

    # Skip compositing for near-square originals
    long_side  = max(orig_w, orig_h)
    short_side = max(min(orig_w, orig_h), 1)
    if (long_side / short_side) < 1.15:
        logger.info(f"Original {orig_w}x{orig_h} is ~square — skipping compositor")
        return video_path

    crop_w = crop_x2 - crop_x1
    crop_h = crop_y2 - crop_y1

    logger.info(
        f"COMPOSITOR: bbox ({crop_x1},{crop_y1})->({crop_x2},{crop_y2}) "
        f"crop={crop_w}x{crop_h} canvas={orig_w}x{orig_h}"
    )

    background = cv2.imread(original_image_path)
    if background is None:
        logger.warning(f"Could not read original image: {original_image_path}")
        return video_path

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    raw_out_path = video_path.replace(".mp4", "_comp_raw.mp4")

    writer = cv2.VideoWriter(
        raw_out_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (orig_w, orig_h)
    )

    # Feather-blend alpha mask for natural edge transitions
    alpha = np.ones((crop_h, crop_w), dtype=np.float32)
    feather_px = max(min(crop_w, crop_h) // 12, 6)
    for px in range(feather_px):
        w = px / feather_px
        alpha[px, :]     *= w
        alpha[-(px+1), :] *= w
        alpha[:, px]     *= w
        alpha[:, -(px+1)] *= w
    alpha_3ch = alpha[:, :, np.newaxis]

    frame_count = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        resized = cv2.resize(frame, (crop_w, crop_h), interpolation=cv2.INTER_LANCZOS4)
        canvas  = background.copy()
        roi     = canvas[crop_y1:crop_y2, crop_x1:crop_x2].astype(np.float32)
        blended = (resized.astype(np.float32) * alpha_3ch + roi * (1.0 - alpha_3ch)).astype(np.uint8)
        canvas[crop_y1:crop_y2, crop_x1:crop_x2] = blended
        writer.write(canvas)
        frame_count += 1

    cap.release()
    writer.release()

    # Re-mux: take video stream from composite, audio stream from original generated clip
    final_out_path = raw_out_path.replace(".mp4", "_final.mp4")
    subprocess.run(
        ["ffmpeg", "-y",
         "-i", raw_out_path,
         "-i", video_path,
         "-c:v", "libx264", "-c:a", "aac",
         "-map", "0:v", "-map", "1:a",
         "-shortest", final_out_path],
        capture_output=True
    )

    if os.path.exists(final_out_path) and os.path.getsize(final_out_path) > 0:
        os.replace(final_out_path, video_path)
        if os.path.exists(raw_out_path):
            os.remove(raw_out_path)
        logger.info(f"Compositor: {frame_count} frames -> {orig_w}x{orig_h}")
        return video_path

    if os.path.exists(raw_out_path):
        os.replace(raw_out_path, video_path)
    return video_path

'''

ga_src = ga_src[:first_def_idx] + compositor_fn + ga_src[first_def_idx:]

with open(GRADIO_PATH, "w") as fh:
    fh.write(ga_src)
print("✅ PATCH C4 — _composite_to_original() injected into gradio_app.py")

# ── Verification ──────────────────────────────────────────────────────────────
print("\n━" * 28)
print("  Patch verification")
print("━" * 55)

with open(GRADIO_PATH,   "r") as fh: ga_final = fh.read()
with open(FACECROP_PATH, "r") as fh: fc_final = fh.read()
with open(HANDLER_PATH,  "r") as fh: hd_final = fh.read()

checks = [
    ("def _composite_to_original"         in ga_final, "Compositor function in gradio_app"),
    ("_composite_to_original(final_video" in ga_final, "Compositor in run_inference"),
    ("_composite_to_original(save_path"   in ga_final, "Compositor in run_multi_gpu"),
    ("_talkdrive_crop_bbox.json"          in ga_final, "gradio_app reads correct bbox file"),
    ("_talkdrive_crop_bbox.json"          in fc_final, "facecrop.py writes correct bbox file"),
    ("/content/models/"                   in ga_final, "Absolute model paths"),
    ('value="lite"'                       in ga_final, "Default mode = lite"),
    ("share=True"                         in ga_final, "Share mode enabled"),
    ("mediapipe" not in hd_final,                      "CPUFaceHandler: no mediapipe"),
    ("cv2.dnn"                            in hd_final, "CPUFaceHandler: OpenCV DNN"),
]

all_ok = True
for ok, label in checks:
    icon = "✅" if ok else "❌"
    if not ok: all_ok = False
    print(f"  {icon}  {label}")

print(f"\n  {'✅ All patches applied.' if all_ok else '❌ Some patches failed — review above.'}")

print("\n── CPUFaceHandler import test ──")
try:
    import sys
    for k in list(sys.modules.keys()):
        if "face_handler" in k or "facecrop" in k:
            del sys.modules[k]
    from flash_head.utils.cpu_face_handler import CPUFaceHandler
    h = CPUFaceHandler()
    print(f"  ✅ Initialised OK  (DNN={h.use_dnn}, Haar={h.use_haar})")
except Exception as err:
    print(f"  ❌ Import failed: {err}")


🔄 Resetting files to upstream state...
✅ PATCH A — CPUFaceHandler rewritten (OpenCV DNN/Haar, no mediapipe)
✅ PATCH B — facecrop.py now persists bbox to /tmp/_talkdrive_crop_bbox.json
✅ PATCH C1 — Gradio header replaced with TalkDrive branding
✅ PATCH C2 — Compositor wired into run_inference()
✅ PATCH C3 — Compositor wired into run_multi_gpu_inference()
✅ PATCH C4 — _composite_to_original() injected into gradio_app.py

━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
  Patch verification
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅  Compositor function in gradio_app
  ✅  Compositor in run_inference
  ✅  Compositor in run_multi_gpu
  ✅  gradio_app reads correct bbox file
  ✅  facecrop.py writes correct bbox file
  ✅  Absolute model paths
  ✅  Default mode = lite
  ✅  Share mode enabled
  ❌  CPUFaceHandler: no mediapipe
  ✅  CPUFaceHandler: OpenCV DNN

  ❌ Some patches failed — review above.

── CPUFaceHandler import test ──
  ✅ Initialised OK  (DNN=True, Haar=False

In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  STAGE 5b — Face-Crop Default Verification & Safety Fix         │
# │                                                                 │
# │  The compositor needs face-crop ON to obtain a bbox file.       │
# │  This cell does a targeted regex check and fix in case the      │
# │  Stage 5 regex landed differently due to whitespace variations. │
# │  Safe to skip if Stage 5 printed all ✅.                        │
# └─────────────────────────────────────────────────────────────────┘

# @title 🛡️ Stage 5b · Safety Check — Face-Crop Default

import re

GRADIO_PATH = "/content/SoulX-FlashHead/gradio_app.py"

with open(GRADIO_PATH, "r") as fh:
    src = fh.read()

patched = re.sub(
    r'(use_face_crop_input\s*=\s*gr\.Checkbox\([^)]*?)value\s*=\s*False',
    r'\1value=True',
    src
)

with open(GRADIO_PATH, "w") as fh:
    fh.write(patched)

result = re.search(r'use_face_crop_input.*?value\s*=\s*(True|False)', patched, re.DOTALL)
if result:
    val = result.group(1)
    print(f"{'✅' if val == 'True' else '❌'}  use_face_crop_input default = {val}")
else:
    print("⚠️  Could not locate use_face_crop_input — check gradio_app.py manually")


✅  use_face_crop_input default = True


In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  STAGE 6 — Launch Gradio Interface                              │
# │                                                                 │
# │  After a few seconds you will see two URLs:                     │
# │   • Local:  http://127.0.0.1:7860  (Colab VM only)              │
# │   • Public: https://xxxxxxxx.gradio.live  (shareable, ~72h)    │
# │                                                                 │
# │  How to use the UI:                                             │
# │   1. Upload a portrait photo (front-facing, good lighting)      │
# │   2. Upload an audio file (WAV/MP3, clear speech)               │
# │   3. Choose Lite (fast) or Pro (quality)                        │
# │   4. Keep Face Crop ON for best results                         │
# │   5. Click Generate — wait ~30–90 seconds on T4                 │
# │   6. Download the output video from the result panel            │
# │                                                                 │
# │  Tips:                                                          │
# │   • Front-facing portraits with neutral backgrounds work best   │
# │   • Clear speech audio produces the best lip-sync              │
# │   • First run loads model into VRAM — subsequent runs faster    │
# └─────────────────────────────────────────────────────────────────┘

# @title 🚀 Stage 6 · Launch TalkDrive UI

%cd /content/SoulX-FlashHead

print("━" * 55)
print("  TalkDrive — Launching Gradio Interface")
print("  Bilal Ansari · ansaribilal.com")
print("━" * 55)
print()
print("⏳ Starting... watch for the public gradio.live URL below:")
print()

!python gradio_app.py

/content/SoulX-FlashHead
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TalkDrive — Launching Gradio Interface
  Bilal Ansari · ansaribilal.com
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

⏳ Starting... watch for the public gradio.live URL below:

2026-04-22 07:38:43.154107: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776843523.175379    2724 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776843523.183194    2724 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776843523.202710    2724 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more th

---

<div style="background: linear-gradient(120deg, #0f2027, #203a43, #2c5364); border-radius: 14px; padding: 28px; text-align:center; margin-top:12px;">

### ✅ TalkDrive — Run Complete

**Quick reference:**

| Step | Action |
|---|---|
| 1 | Upload a front-facing portrait photo |
| 2 | Upload a speech audio file (WAV/MP3) |
| 3 | Select Lite (quick) or Pro (quality) |
| 4 | Keep Face Crop ON |
| 5 | Hit Generate and wait ~30–90s |
| 6 | Download your result video |

---

**Model credit:** [Soul-AILab/SoulX-FlashHead](https://github.com/Soul-AILab/SoulX-FlashHead)  
**Notebook author:** Bilal Ansari · [ansaribilal.com](https://ansaribilal.com)

</div>
